<a href="https://colab.research.google.com/github/Khoawawa/DeepLearning-HCMUT-ASS-MS/blob/main/ImageClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import torch.nn as nn

In [19]:
batch_size = 64

In [20]:
import torch
import torchvision
import torchvision.transforms as transforms

IMAGENET_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
trainset = torchvision.datasets.CIFAR10(root='/content/data', train=True,
                                        download=True, transform=IMAGENET_transform)
train_size = int(0.9 * len(trainset))
val_size = len(trainset) - train_size

generator = torch.Generator().manual_seed(42)

trainset, valset = torch.utils.data.random_split(
    trainset,
    [train_size, val_size],
    generator=generator
)
# Create a DataLoader for the training set
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size,
                                          shuffle=False, num_workers=2)
# Load the test dataset
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=IMAGENET_transform)

# Create a DataLoader for the test set
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('airplane', 'automobile', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [31]:
from tqdm import tqdm
import torch
import copy

def train(model, trainloader, valloader, optimizer, criterion, device, num_epochs, patience=3):
    model.to(device)

    best_val_loss = float("inf")
    best_model_wts = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        train_loop = tqdm(trainloader, leave=True)

        for inputs, labels in train_loop:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            train_loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
            train_loop.set_postfix(train_loss=loss.item())

        train_loss = running_loss / len(trainloader)

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in valloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_loss /= len(valloader)
        val_acc = correct / total

        print(
            f"Epoch {epoch+1}: "
            f"Train Loss={train_loss:.4f} "
            f"Val Loss={val_loss:.4f} "
            f"Val Acc={val_acc:.4f}"
        )
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0

            torch.save(best_model_wts, "best_model.pth")

        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping triggered")
            break
    # load best weights
    model.load_state_dict(best_model_wts)

    return model

In [32]:
def test(model, testloader, device):
  model.eval()
  model.to(device)

  correct = 0
  total = 0

  all_outputs = []
  all_labels = []

  with torch.no_grad():
    for inputs, labels in tqdm(testloader):
      inputs = inputs.to(device)
      labels = labels.to(device)

      outputs = model(inputs)

      _, preds = torch.max(outputs, 1)

      correct += (preds == labels).sum().item()
      total += labels.size(0)

      all_outputs.append(outputs.cpu())
      all_labels.append(labels.cpu())

  accuracy = correct / total

  all_outputs = torch.cat(all_outputs)
  all_labels = torch.cat(all_labels)

  return accuracy, all_outputs, all_labels


In [33]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

def evaluation(accuracy, outputs, labels):
  probs = torch.softmax(outputs, dim=1).numpy()
  preds = torch.argmax(outputs, dim=1).numpy()
  labels = labels.numpy()

  cm = confusion_matrix(labels, preds)

  plt.figure()
  plt.imshow(cm)
  plt.title("Confusion Matrix")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.colorbar()
  plt.show()

  precision = precision_score(labels, preds, average="macro")
  recall = recall_score(labels, preds, average="macro")
  f1 = f1_score(labels, preds, average="macro")
  print("Accuracy:", accuracy)
  print("Precision:", precision)
  print("Recall:", recall)
  print("F1:", f1)

  print(classification_report(labels, preds))

In [34]:
def pipeline(model, trainloader, valloader, testloader, optimizer, criterion, device, num_epochs):
  best_model = train(model, trainloader, valloader, optimizer, criterion, device, num_epochs)
  accuracy, all_outputs, all_labels = test(best_model, testloader, device)
  evaluation(accuracy, all_outputs, all_labels)
  return best_model, accuracy

In [35]:
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [36]:
from torchvision.models import resnet18, ResNet18_Weights

resnet_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)

# Freeze backbone
for param in resnet_model.parameters():
    param.requires_grad = False

# Unfreeze classifier
for param in resnet_model.fc.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(resnet_model.fc.parameters(), lr=1e-3)

In [ ]:
pipeline(resnet_model, trainloader, valloader,testloader,optimizer, nn.CrossEntropyLoss(), device, 20)

Epoch [1/20]:  28%|██▊       | 195/704 [00:26<01:02,  8.20it/s, train_loss=0.858]

In [ ]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, num_classes)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
pipeline(model, trainloader, valloader,testloader,optimizer, nn.CrossEntropyLoss(), device, 20)

In [ ]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

model.fc = nn.Linear(model.fc.in_features, num_classes)

optimizer = torch.optim.Adam([
    {"params": model.layer1.parameters(), "lr": 1e-5},
    {"params": model.layer2.parameters(), "lr": 1e-5},
    {"params": model.layer3.parameters(), "lr": 5e-5},
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
])

In [ ]:
pipeline(model, trainloader, valloader,testloader,optimizer, nn.CrossEntropyLoss(), device, 20)